# Phase 2 - Orders, Customers, and Inventory Analysis
Notebook nay thuc hien cac phan tich tru cot A/B: RFM, cohort retention, LTV:CAC, doanh thu theo nam, category matrix, return cost.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED = ROOT / 'data' / 'processed'

fact_transactions = pd.read_parquet(PROCESSED / 'fact_transactions.parquet')
dim_users = pd.read_parquet(PROCESSED / 'dim_users.parquet')
dim_products = pd.read_parquet(PROCESSED / 'dim_products.parquet')

fact_transactions['created_at'] = pd.to_datetime(fact_transactions['created_at'], errors='coerce')
fact_transactions['year'] = fact_transactions['created_at'].dt.year

print(f'fact_transactions: {fact_transactions.shape}')
print(f'dim_users: {dim_users.shape}')
print(f'dim_products: {dim_products.shape}')

In [ ]:
rfm = dim_users[['user_id', 'Recency', 'Frequency', 'Monetary', 'return_rate_pct', 'rfm_segment']].copy()
rfm = rfm.dropna(subset=['Recency', 'Frequency', 'Monetary', 'rfm_segment'])

segment_dist = rfm['rfm_segment'].value_counts().reset_index()
segment_dist.columns = ['rfm_segment', 'user_count']
segment_dist['pct'] = (segment_dist['user_count'] / len(rfm) * 100).round(2)
segment_dist.to_csv(PROCESSED / 'rfm_segment_analysis.csv', index=False)

fig = px.bar(
    segment_dist,
    x='rfm_segment',
    y='user_count',
    title='RFM Segment Distribution',
    text='pct'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.show()

segment_dist

In [ ]:
orders = fact_transactions[['user_id', 'created_at']].dropna().copy()
users_signup = dim_users[['user_id']].copy()

if 'created_at' in dim_users.columns:
    users_signup['signup_year'] = pd.to_datetime(dim_users['created_at'], errors='coerce').dt.year
else:
    # Fallback: when signup timestamp is unavailable in dim_users output
    users_signup['signup_year'] = orders.groupby('user_id')['created_at'].min().dt.year

orders['purchase_year'] = orders['created_at'].dt.year
cohort = users_signup.merge(orders[['user_id', 'purchase_year']], on='user_id', how='left')
cohort = cohort.dropna(subset=['signup_year', 'purchase_year'])
cohort['signup_year'] = cohort['signup_year'].astype(int)
cohort['purchase_year'] = cohort['purchase_year'].astype(int)

cohort_counts = cohort.groupby(['signup_year', 'purchase_year'])['user_id'].nunique().reset_index(name='users')
base = cohort_counts[cohort_counts['signup_year'] == cohort_counts['purchase_year']][['signup_year', 'users']].rename(columns={'users': 'base_users'})
cohort_retention = cohort_counts.merge(base, on='signup_year', how='left')
cohort_retention['retention_pct'] = (cohort_retention['users'] / cohort_retention['base_users'] * 100).round(2)
cohort_retention.to_csv(PROCESSED / 'cohort_retention_analysis.csv', index=False)

heat = cohort_retention.pivot(index='signup_year', columns='purchase_year', values='retention_pct').fillna(0)
fig = px.imshow(heat, text_auto='.1f', aspect='auto', color_continuous_scale='RdYlGn',
                title='Cohort Retention (%)')
fig.show()

cohort_retention.head()

In [ ]:
one_time = rfm[rfm['Frequency'] == 1]
ltv_majority = float(one_time['Monetary'].mean() * 0.519) if len(one_time) else np.nan
ltv_all = float(rfm['Monetary'].mean() * 0.519) if len(rfm) else np.nan
assumed_cac = 40.0

ltv_cac_df = pd.DataFrame([
    {'segment': 'one_time_buyers', 'ltv': ltv_majority, 'assumed_cac': assumed_cac, 'ltv_cac_ratio': ltv_majority / assumed_cac if pd.notna(ltv_majority) else np.nan},
    {'segment': 'all_users', 'ltv': ltv_all, 'assumed_cac': assumed_cac, 'ltv_cac_ratio': ltv_all / assumed_cac if pd.notna(ltv_all) else np.nan},
])
ltv_cac_df.to_csv(PROCESSED / 'ltv_cac_analysis.csv', index=False)
ltv_cac_df

In [ ]:
annual = (
    fact_transactions.dropna(subset=['year'])
    .groupby('year', as_index=False)
    .agg(
        total_revenue=('sale_price', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        num_orders=('order_id', 'nunique'),
        num_items=('order_item_id', 'count')
    )
)
annual = annual[annual['year'] <= 2023].sort_values('year').copy()
annual['aov'] = annual['total_revenue'] / annual['num_orders']
annual['gross_margin_pct'] = annual['gross_profit'] / annual['total_revenue'] * 100
annual['yoy_growth_pct'] = annual['total_revenue'].pct_change() * 100
annual.to_csv(PROCESSED / 'annual_revenue_analysis.csv', index=False)

fig = go.Figure()
fig.add_trace(go.Bar(x=annual['year'], y=annual['total_revenue'], name='Revenue'))
fig.add_trace(go.Scatter(x=annual['year'], y=annual['gross_margin_pct'], mode='lines+markers',
                         name='Gross Margin %', yaxis='y2'))
fig.update_layout(
    title='Revenue and Gross Margin by Year',
    yaxis=dict(title='Revenue'),
    yaxis2=dict(title='Gross Margin %', overlaying='y', side='right')
)
fig.show()

annual

In [ ]:
category = (
    fact_transactions.merge(dim_products[['product_id', 'category']], on='product_id', how='left')
    .groupby('category', as_index=False)
    .agg(
        revenue=('sale_price', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        num_items=('order_item_id', 'count'),
        num_returned=('is_returned', 'sum')
    )
)
category['gross_margin_pct'] = category['gross_profit'] / category['revenue'] * 100
category['return_rate_pct'] = category['num_returned'] / category['num_items'] * 100

median_revenue = category['revenue'].median()
median_margin = category['gross_margin_pct'].median()

def classify(row):
    if row['revenue'] > median_revenue and row['gross_margin_pct'] > median_margin:
        return 'Star'
    if row['revenue'] > median_revenue and row['gross_margin_pct'] <= median_margin:
        return 'Cash Cow'
    if row['revenue'] <= median_revenue and row['gross_margin_pct'] > median_margin:
        return 'Problem Child'
    return 'Dog'

category['quadrant'] = category.apply(classify, axis=1)
category.to_csv(PROCESSED / 'category_performance_analysis.csv', index=False)

fig = px.scatter(
    category,
    x='revenue',
    y='gross_margin_pct',
    size='num_items',
    color='return_rate_pct',
    hover_name='category',
    title='Category Performance Matrix'
)
fig.show()

category.sort_values('revenue', ascending=False).head(10)

In [ ]:
returns = fact_transactions[fact_transactions['is_returned'] == 1].merge(
    dim_products[['product_id', 'category']], on='product_id', how='left'
)
returns_by_cat = returns.groupby('category', as_index=False).agg(
    returned_items=('order_item_id', 'count'),
    return_value=('sale_price', 'sum')
)
returns_by_cat['estimated_processing_cost'] = returns_by_cat['returned_items'] * 20
returns_by_cat.to_csv(PROCESSED / 'return_cost_analysis.csv', index=False)

total_return_items = int(fact_transactions['is_returned'].sum())
total_return_pct = float(fact_transactions['is_returned'].mean() * 100)
total_estimated_cost = float(total_return_items * 20)

print(f'Total returned items: {total_return_items:,}')
print(f'Return rate: {total_return_pct:.2f}%')
print(f'Estimated processing cost: ${total_estimated_cost:,.0f}')

returns_by_cat.sort_values('return_value', ascending=False).head(10)

## Key Takeaways
- RFM segmentation and cohort retention available in exported CSV files.
- Revenue grows strongly while gross margin trend can be inspected yearly.
- Category matrix highlights high-volume and high-margin groups for prioritization.
- Return processing cost is estimated and ready for dashboard integration.